# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
print("Hello World!")


Hello World!


In [3]:
pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 106.3 kB/s eta 0:00:0000:0100:03
Note: you may need to restart the kernel to use updated packages.


In [12]:
import gurobipy as gp

# Data
n_var = 3
n_con = 2
ubs = [10, gp.GRB.INFINITY, gp.GRB.INFINITY]
c=[1,2,5]
A = [[-1,1,3],[1,3,-7]]
b = [-5,10]

# Model
m = gp.Model("LP")
# x1 = m.addVar(lb=0,ub=10,vtype=gp.GRB.CONTINUOUS,name="x1")
# x2 = m.addVar(lb=0,ub=gp.GRB.INFINITY,vtype=gp.GRB.CONTINUOUS,name="x2")
# x3 = m.addVar(lb=0,ub=gp.GRB.INFINITY,vtype=gp.GRB.CONTINUOUS,name="x3")
# n_var = 3
# n_con = 2
# ubs = [10, gp.GRB.INFINITY, gp.GRB.INFINITY]
x = m.addVars(n_var,lb=0,ub=ubs,vtype=gp.GRB.CONTINUOUS,name='x')
# m.setObjective(x1+2*x2+5*x3, gp.GRB.MAXIMIZE)
# c=[1,2,5]
# m.setObjective(sum(c[j]*x[j] for j in range(n_var)), gp.GRB.MAXIMIZE)
m.setObjective(x.prod(c), gp.GRB.MAXIMIZE)
# m.addConstr(-x1+x2+3*x3<=-5,name='con1')
# m.addConstr(x1+3*x2-7*x3<=10,name='con2')
# A = [[-1,1,3],[1,3,-7]]
# b = [-5,10]
# m.addConstrs((gp.quicksum(A[i][j]*x[j] for j in range(n_var)) <= b[i] for i in range(n_con)),name='con')
m.addConstrs((x.prod(A[i]) <= b[i] for i in range(n_con)),name='con')
m.optimize()
print("Optimal value:", m.objVal)
print("Optimal solution:",[x[j].x for j in range(n_var)])

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.4.0 24E263)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

Optimize a model with 2 rows, 3 columns and 6 nonzeros
Model fingerprint: 0xf6b9ddf6
Coefficient statistics:
  Matrix range     [1e+00, 7e+00]
  Objective range  [1e+00, 5e+00]
  Bounds range     [1e+01, 1e+01]
  RHS range        [5e+00, 1e+01]
Presolve time: 0.00s
Presolved: 2 rows, 3 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    9.0000000e+30   1.250000e+30   9.000000e+00      0s
       2    1.9062500e+01   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.906250000e+01
Optimal value: 19.0625
Optimal solution: [10.0, 2.1875, 0.9375]


In [18]:
from gurobipy import *

# Data
plants = ['1','2','3']
retailers = ['A','B','C','D']
capacity_raw = [1700,2000,1700]
# capacity = {plants[i]: capacity_raw[i] for i in range(len(capacity_raw))}
capacity = dict(zip(plants,capacity_raw))
demand_raw = [1700,1000,1500,1200]
# demand = {retailers[i]: demand_raw[i] for i in range(len(demand_raw))}
demand = dict(zip(retailers,demand_raw))
cost_raw = [[5,3,2,6],[7,7,8,10],[6,5,3,8]]
cost = {(plants[i],retailers[j]): cost_raw[i][j] for i in range(len(plants)) for j in range(len(retailers))}
n_plants = len(plants)
n_retailers = len(retailers)

# Model
m = Model("transportation")
x = m.addVars(plants, retailers, vtype=GRB.CONTINUOUS, name="x")
# m.setObjective(quicksum(cost[i][j]*x[i,j] for i in range(n_plants) for j in range(n_retailers)), GRB.MINIMIZE)
m.setObjective(x.prod(cost), GRB.MINIMIZE)
# m.addConstrs(quicksum(x[i,j] for i in range(n_plants)) == demand[j] for j in range(n_retailers))
# m.addConstrs(quicksum(x[i,j] for i in plants) == demand[j] for j in retailers)
m.addConstrs(x.sum('*',j) == demand[j] for j in retailers)
# m.addConstrs(quicksum(x[i,j] for j in range(n_retailers)) <= capacity[i] for i in range(n_plants))
# m.addConstrs(quicksum(x[i,j] for j in retailers) <= capacity[i] for i in plants)
m.addConstrs(x.sum(i,'*') <= capacity[i] for i in plants)
m.optimize()

# Output
print("Optimal value:", m.objVal)
for i in plants:
    for j in retailers:
        if x[i, j].x > 0:
            print(f"Ship {x[i, j].x} units from plant {i} to retailer {j}")

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.4.0 24E263)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

Optimize a model with 7 rows, 12 columns and 24 nonzeros
Model fingerprint: 0x0164a657
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 1e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+03, 2e+03]
Presolve time: 0.00s
Presolved: 7 rows, 12 columns, 24 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.1700000e+04   3.700000e+03   0.000000e+00      0s
       5    2.8200000e+04   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.820000000e+04
Optimal value: 28200.0
Ship 500.0 units from plant 1 to retailer B
Ship 1200.0 units from plant 1 to retailer D
Ship 1700.0 units from plant 2 to retailer A
Ship 300.0 units from plant 2 to retailer B
Ship 200.0 